# Notebook 01 – Bronze Layer

**Why Bronze Layer?**

The Bronze layer is designed to preserve the original source data exactly as received.

Its responsibilities are:

- Landing raw data
- Supporting incremental ingestion
- Maintaining historical records
- Providing reliable input for Silver transformations

No business logic, cleansing, or standardization is applied here.

**Objective**

The objective of this notebook is to ingest raw healthcare data from multiple source systems into the Bronze layer of the Medallion Architecture without modifying the original data.

The Bronze layer acts as the immutable landing zone where all incoming data is stored in Delta format for downstream processing.

**Data Sources**
This notebook ingests data from three different healthcare formats:

- FHIR R4 JSON
- HL7 ADT messages
- Flat CSV files



### 1. Unity Catalog Bootstrap



In [0]:
%run ./00_UnityCatalog_Bootstrap_&_Audit_log

This notebook initializes the environment by loading Unity Catalog configurations, storage credentials, and audit logging utilities used throughout the project.

### 2. Import Libraries
### 
Required Python libraries such as

- datetime
- PySpark

are imported for timestamp generation and Spark processing.

In [0]:
from datetime import datetime

start_time = datetime.now()

##  **FHIR Data Ingestion** 

### 3. Read FHIR JSON Files

In [0]:
rawRoot = dbutils.secrets.get(
    scope="healthcare",
    key="raw-root"
)

rawSourcePathFhir = rawRoot + "fhir/"


In [0]:
df = spark.read.option("multiline","true").json(rawSourcePathFhir + "*.json")

df.show(5)

In [0]:
df.printSchema()

In [0]:
df.count()

%md
Reads raw FHIR JSON files directly from ADLS Gen2.

Purpose:

- Validate connectivity
- Inspect schema
- Count records
- Verify incoming data

No transformations are performed.

### 4. JSON TO DELTA

In [0]:
bronzeRoot = dbutils.secrets.get(
    scope="healthcare",
    key="bronze-root"
)

bronze_fhir = bronzeRoot + "fhir_raw/"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.fhir_raw
USING DELTA
LOCATION '{bronze_fhir}'
""")

In [0]:
%sql
SHOW TABLES IN db_healthcare_kl.bronze;

Creates the Bronze Delta table:

bronze.fhir_raw

This stores the original JSON payload exactly as received.

Benefits:

- ACID transactions
- Time Travel
- Schema evolution
- Reliable storage

### 5. Auto Loader

In [0]:

schema_path = bronzeRoot + "schema/fhir"

checkpoint_path = bronzeRoot + "checkpoints/fhir"

bronze_output_path = bronze_fhir

In [0]:
schema = (
    spark.read
    .option("multiLine","true")
    .json(rawSourcePathFhir)
    .schema
)

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .schema(schema)
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(rawSourcePathFhir)
    .withColumn(
        "ingestion_time",
        current_timestamp()
    )
)

In [0]:

bronze_df.writeStream\
        .format("delta")\
        .option(
            "checkpointLocation",
            checkpoint_path
        )\
        .trigger(availableNow=True)\
        .start(bronze_output_path)


In [0]:
bronze_table = spark.read.table("bronze.fhir_raw")

bronze_table.printSchema()

%md
Purpose:

- Detect newly arrived files automatically
- Avoid reprocessing old files
- Support incremental ingestion
- Scale to large datasets

Auto Loader also maintains:

- Schema location
- Checkpoint location

to ensure fault-tolerant streaming.

### 7. Validate Bronze Table

Uses commands like

- DESCRIBE DETAIL
- DESCRIBE HISTORY

to verify:

- Table metadata
- Delta version history
- Storage location
- Transaction log

In [0]:
%sql
DESCRIBE DETAIL db_healthcare_kl.bronze.fhir_raw;

In [0]:
%sql
DESCRIBE HISTORY db_healthcare_kl.bronze.fhir_raw;

## **HL7 Data Ingestion**

The same ingestion pattern is followed for HL7 files.

- Read HL7 files
    - Reads raw HL7 ADT messages from ADLS.

- Auto Loader
    - Processes only newly arrived HL7 files.

- Store into
    - bronze.hl7_adt_raw

- No parsing or cleaning is done in Bronze.

In [0]:
%sql
DROP TABLE IF EXISTS db_healthcare_kl.bronze.hl7_adt_raw;

In [0]:
raw_path = rawRoot + "hl7/ADT"

checkpoint_path = bronzeRoot + "checkpoints/hl7_adt"

bronze_output_path = bronzeRoot + "hl7_adt_raw"

In [0]:
schema = (
    spark.read
    .text(raw_path)
    .schema
)

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .load(raw_path)
        .selectExpr(
            "path",
            "cast(content as string) as raw_message"
        )
        .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
(
    bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start(bronze_output_path)
        .awaitTermination()
)

In [0]:
spark.sql(f"""
CREATE TABLE db_healthcare_kl.bronze.hl7_adt_raw
USING DELTA
LOCATION '{bronze_output_path}'
""")

In [0]:
%sql
DESCRIBE db_healthcare_kl.bronze.hl7_adt_raw;

In [0]:
df = spark.read.table("db_healthcare_kl.bronze.hl7_adt_raw")

df.select("path", "raw_message").show(truncate=False)

## **Flat Files Data Ingestion**

A reusable function
    **ingest_to_bronze()**
is created to ingest multiple CSV datasets.

It is then used for tables such as:

- patients
- encounters
- conditions
- observations
- medications

Each dataset is stored as a separate Bronze Delta table.

In [0]:
def ingest_to_bronze(table_name):
    
    raw_path = rawRoot + f"flat files/{table_name}"

    checkpoint_path = bronzeRoot + f"checkpoints/{table_name}"

    bronze_output_path = bronzeRoot + f"{table_name}_bronze"

    schema = (
        spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(raw_path)
            .schema
    )

    bronze_df = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .schema(schema)
            .load(raw_path)
            .withColumn("ingestion_timestamp", current_timestamp())
    )

    (
        bronze_df.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_path)
            .trigger(availableNow=True)
            .start(bronze_output_path)
            .awaitTermination()
    )

    print(f"Finished loading {table_name}")

In [0]:
tables = [
    "patients",
    "encounters",
    "conditions",
    "observations",
    "medications"
]

for tbl in tables:
    ingest_to_bronze(tbl)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.patients_raw
USING DELTA
LOCATION '{bronzeRoot}patients_bronze'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.encounters_raw
USING DELTA
LOCATION '{bronzeRoot}encounters_bronze'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.conditions_raw
USING DELTA
LOCATION '{bronzeRoot}conditions_bronze'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.observations_raw
USING DELTA
LOCATION '{bronzeRoot}observations_bronze'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS db_healthcare_kl.bronze.medications_raw
USING DELTA
LOCATION '{bronzeRoot}medications_bronze'
""")

In [0]:
from datetime import datetime

end_time = datetime.now()

conditions_raw = spark.read.table("db_healthcare_kl.bronze.conditions_raw")
encounters_raw = spark.read.table("db_healthcare_kl.bronze.encounters_raw")
observations_raw = spark.read.table("db_healthcare_kl.bronze.observations_raw")
medications_raw = spark.read.table("db_healthcare_kl.bronze.medications_raw")
patients_raw = spark.read.table("db_healthcare_kl.bronze.patients_raw")
fhir_raw = spark.read.table("db_healthcare_kl.bronze.fhir_raw")
hl7_raw = spark.read.table("db_healthcare_kl.bronze.hl7_adt_raw")

record_count = (
    conditions_raw.count() +
    encounters_raw.count() +
    observations_raw.count() +
    medications_raw.count() +
    patients_raw.count() +
    fhir_raw.count() +
    hl7_raw.count()
)

log_pipeline_run(
    pipeline_name="Healthcare Lakehouse",
    layer="Bronze",
    status="SUCCESS",
    records_processed=record_count,
    start_time=start_time,
    end_time=end_time
)

**Bronze Tables Created**

This notebook creates the following Bronze tables:

- bronze.fhir_raw
- bronze.hl7_adt_raw
- bronze.patients_raw
- bronze.encounters_raw
- bronze.conditions_raw
- bronze.observations_raw
- bronze.medications_raw